# Causal Inference in Practice
## Week 12 — Double / Debiased Machine Learning · Practice Notebook

> **Block IV — Modern methods & application**
>
> Use flexible ML for the nuisance parts without letting it bias the effect you care about.

**How to use this notebook.** Run the cells top to bottom. Sections marked
**🔧 Exercise** contain a `# TODO` for you to complete; a matching
**✅ Solution** cell follows (collapsed in spirit — try it yourself first).
Every dataset here is *simulated with a known ground truth*, so you can always
check whether your estimate recovered the right answer.

*Estimated time: 60–90 minutes. Toolkit: `numpy`, `pandas`, `statsmodels`,
`scikit-learn`, `matplotlib` — all standard.*

---


In [ ]:
# --- Environment check & shared setup -------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (7, 4.2), "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 11})

RNG = np.random.default_rng(7)   # one seed for the whole notebook → reproducible
print("Environment OK — numpy", np.__version__, "| pandas", pd.__version__)

## 1 · A partially linear model with a known effect

We simulate the workhorse of this week:

$$Y = \theta\,D + g(X) + \varepsilon, \qquad D = m(X) + v$$

with **nonlinear** nuisance functions `g, m`, a **high-dimensional** covariate matrix `X` (20 columns), and a **known** true effect `θ = 0.8`. The covariates `X` confound: they drive both the treatment `D` (through `m`) and the outcome `Y` (through `g`). Because the same nonlinear `common(X)` enters both, a naive analysis will be badly biased — and that is the point.

We reuse the provided `RNG` (seeded once at setup); we never reseed.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import statsmodels.api as sm

TRUE_THETA = 0.8
n, p = 3000, 20

def simulate(n, p):
    X = RNG.uniform(-1.5, 1.5, size=(n, p))
    # one nonlinear function drives BOTH treatment and outcome
    common = (1.5*np.maximum(X[:, 0], 0)   # kink in X0
              + X[:, 1]**2                 # curvature in X1
              + 0.8*np.sin(2*X[:, 2]))     # wiggle in X2
    g = common                              # outcome nuisance g(X)
    m = common                              # treatment nuisance m(X)
    D = m + RNG.normal(size=n)              # D = m(X) + v
    Y = TRUE_THETA*D + g + RNG.normal(size=n)  # Y = theta*D + g(X) + e
    return X, D, Y

X, D, Y = simulate(n, p)
print(f'n = {n}, p = {p} covariates,  TRUE theta = {TRUE_THETA}')
print(f'D ranges [{D.min():.2f}, {D.max():.2f}],  '
      f'corr(D, common-driver) is large by construction')

## 2 · The naive estimates are biased

Two tempting naive analyses, both wrong:

1. **OLS of Y on D and (linear) X** — linear terms cannot absorb the nonlinear `g(X)`, so leftover confounding inflates the coefficient on `D`.
2. **A single flexible forest** of `Y` on `(D, X)`, reading off the implied effect of a one-unit bump in `D` — regularization bias pulls it away from the truth.

Both should miss `θ = 0.8`.

In [ ]:
# Naive 1: OLS of Y on D and X (linear) -- arrays, so .params[1] is D
naive_ols = sm.OLS(Y, sm.add_constant(np.c_[D, X])).fit().params[1]

# Naive 2: one forest on (D, X); effect = avg predicted bump from D->D+1
rf_all = RandomForestRegressor(n_estimators=100, max_depth=10,
                               random_state=7).fit(np.c_[D, X], Y)
naive_ml = float((rf_all.predict(np.c_[D + 1.0, X])
                  - rf_all.predict(np.c_[D, X])).mean())

print(f'naive OLS (D + linear X) = {naive_ols:.3f}')
print(f'naive single-forest      = {naive_ml:.3f}')
print(f'TRUE theta               = {TRUE_THETA:.3f}')
assert abs(naive_ols - TRUE_THETA) > 0.2, 'naive OLS should be biased'
assert abs(naive_ml - TRUE_THETA) > 0.1, 'naive plug-in should be biased'

The naive estimates land well above 0.8. The flexible forest predicts `Y` well, yet its implied effect is still biased — **prediction is not estimation.** We need to change *how* we read the effect off, not just *how flexible* the learner is.

## 3 · Frisch–Waugh–Lovell, warmed up with OLS

Before the ML version, confirm the **residual-on-residual** identity on a purely linear toy. FWL says the coefficient on `D` in `Y ~ D + X` equals the slope of (Y residualized on X) on (D residualized on X). We'll verify the two give the *same* number.

In [ ]:
# Linear toy: Y = 0.8*D + X@beta + noise, D = X@gamma + noise
Xl = RNG.normal(size=(4000, 6))
beta  = RNG.normal(size=6)
gamma = RNG.normal(size=6)
Dl = Xl @ gamma + RNG.normal(size=4000)
Yl = 0.8*Dl + Xl @ beta + RNG.normal(size=4000)

# (a) full OLS coefficient on D
full = sm.OLS(Yl, sm.add_constant(np.c_[Dl, Xl])).fit().params[1]

# (b) FWL: residualize Y and D on X (with intercept), regress residuals
Xc = sm.add_constant(Xl)
rY = Yl - sm.OLS(Yl, Xc).fit().predict(Xc)
rD = Dl - sm.OLS(Dl, Xc).fit().predict(Xc)
fwl = sm.OLS(rY, rD).fit().params[0]

print(f'full OLS coef on D       = {full:.4f}')
print(f'residual-on-residual (FWL) = {fwl:.4f}')
assert abs(full - fwl) < 1e-8, 'FWL identity should hold exactly'
print('FWL identity confirmed: partialling X out of both sides is enough.')

DML is exactly this, with the **linear** projections `E[Y|X]`, `E[D|X]` replaced by **flexible ML**. One catch the linear case hides: ML models *overfit* the rows they are trained on, so we must **cross-fit** (Section 5).

## 4 · Partially-linear DML, by hand

Now the real thing on the nonlinear, high-dimensional data:

1. **Cross-fit** flexible models for `E[Y|X]` and `E[D|X]` with K-fold splitting (predict each row using models trained on the *other* folds).
2. Form out-of-fold residuals `Ỹ = Y − Ê[Y|X]`, `D̃ = D − Ê[D|X]`.
3. **Regress `Ỹ` on `D̃`** to get `θ̂`, with a robust standard error.

The estimate should recover `θ = 0.8` with a valid CI.

In [ ]:
from sklearn.model_selection import KFold

def dml_plm(X, D, Y, n_splits=5, crossfit=True,
            n_estimators=100, max_depth=None):
    """Partially-linear DML. Returns theta_hat, SE, and residuals."""
    nn = len(Y)
    rY = np.zeros(nn)
    rD = np.zeros(nn)
    if crossfit:
        for tr, te in KFold(n_splits, shuffle=True,
                            random_state=7).split(X):
            mY = RandomForestRegressor(n_estimators=n_estimators,
                     max_depth=max_depth, random_state=7).fit(X[tr], Y[tr])
            mD = RandomForestRegressor(n_estimators=n_estimators,
                     max_depth=max_depth, random_state=7).fit(X[tr], D[tr])
            rY[te] = Y[te] - mY.predict(X[te])
            rD[te] = D[te] - mD.predict(X[te])
    else:  # in-sample: fit and predict on the SAME rows (overfits!)
        mY = RandomForestRegressor(n_estimators=n_estimators,
                 max_depth=max_depth, random_state=7).fit(X, Y)
        mD = RandomForestRegressor(n_estimators=n_estimators,
                 max_depth=max_depth, random_state=7).fit(X, D)
        rY = Y - mY.predict(X)
        rD = D - mD.predict(X)
    # orthogonal estimate: slope of residual on residual (robust SE)
    fit = sm.OLS(rY, rD).fit(cov_type='HC1')
    return fit.params[0], fit.bse[0], rY, rD

theta_hat, se, rY, rD = dml_plm(X, D, Y, crossfit=True)
lo, hi = theta_hat - 1.96*se, theta_hat + 1.96*se
print(f'DML (cross-fitted) theta_hat = {theta_hat:.3f}  SE = {se:.3f}')
print(f'95% CI = [{lo:.3f}, {hi:.3f}]    TRUE theta = {TRUE_THETA}')
assert abs(theta_hat - TRUE_THETA) < 0.1, 'DML should recover ~0.8'
assert lo < TRUE_THETA < hi, 'CI should cover the truth'

DML recovered the truth and its interval covers `0.8`, while every naive estimate in Section 2 was biased. The orthogonal residual-on-residual score, fed cross-fitted ML nuisances, is what made the difference.

### 🔧 Exercise 4.1 — residualization removes the confounding

If residualizing worked, the **treatment residual** `D̃` should be (nearly) uncorrelated with the confounding driver `common(X)`, even though raw `D` is strongly correlated with it. Compute both correlations and compare.

Fill in the `# TODO`s. The skeleton runs as-is (`...` are placeholders); replace them, then run the solution cell.

In [ ]:
driver = (1.5*np.maximum(X[:, 0], 0) + X[:, 1]**2
          + 0.8*np.sin(2*X[:, 2]))   # the confounding common(X)

corr_raw = np.corrcoef(D, driver)[0, 1]      # large (confounded)

# TODO: correlation of the residualized treatment D~ with the driver
corr_resid = ...   # TODO: np.corrcoef(rD, driver)[0, 1]

print(f'corr(raw D, driver)      = {corr_raw:+.3f}')
# print(f'corr(residual D~, driver) = {corr_resid:+.3f}')

### ✅ Solution 4.1

In [ ]:
corr_resid = np.corrcoef(rD, driver)[0, 1]
print(f'corr(raw D, driver)       = {corr_raw:+.3f}')
print(f'corr(residual D~, driver) = {corr_resid:+.3f}')
assert abs(corr_raw) > 0.5, 'raw D should be confounded by the driver'
assert abs(corr_resid) < 0.15, 'residualization should strip the driver'
print('\nResidualizing D on X removed almost all of the confounding '
      'variation -> what is left is as-good-as-random.')

## 5 · Why cross-fitting matters

Cross-fitting predicts each row using models trained on the *other* folds. The naive alternative fits the nuisance on **all** rows and predicts those same rows — so the forest **memorizes** them and the in-sample residuals are far too small. We compare the two and watch the residual variances **collapse** without cross-fitting.

In [ ]:
theta_nc, se_nc, rY_nc, rD_nc = dml_plm(X, D, Y, crossfit=False)

print(f'{"":22s}{"cross-fit":>12s}{"in-sample":>12s}')
print(f'{"theta_hat":22s}{theta_hat:12.3f}{theta_nc:12.3f}')
print(f'{"reported SE":22s}{se:12.3f}{se_nc:12.3f}')
print(f'{"Var(D residual)":22s}{np.var(rD):12.3f}{np.var(rD_nc):12.3f}')
print(f'{"Var(Y residual)":22s}{np.var(rY):12.3f}{np.var(rY_nc):12.3f}')

# The in-sample residual variances collapse -- the overfitting fingerprint.
assert np.var(rD_nc) < 0.5*np.var(rD), 'in-sample D resid should collapse'
assert np.var(rY_nc) < 0.5*np.var(rY), 'in-sample Y resid should collapse'
print('\nIn-sample residual variances collapsed: the forest memorized the\n'
      'rows, so the in-sample SE is too small and the inference is invalid.')

The point estimates look similar here, but the **in-sample residual variances cratered** (D: ~1.1 → ~0.15; Y: ~1.9 → ~0.27). That is the overfitting fingerprint: the model fit the noise of the very rows it then 'predicts,' so the reported standard error is too small and the asymptotic-normality guarantee is gone. Cross-fitting is what keeps the inference honest — and it is essentially free.

### 🔧 Exercise 5.1 — the in-sample fit is suspiciously perfect

Confirm the overfitting directly: a forest fit on all rows predicts those same rows with an **in-sample** R² close to 1, even though `D = common(X) + noise` has irreducible noise that no honest model could explain. Compute the in-sample R² of the `D`-model.

Complete the `# TODO`.

In [ ]:
mD_full = RandomForestRegressor(n_estimators=100,
                                random_state=7).fit(X, D)

# TODO: in-sample R^2 of the D-model (fit and scored on the SAME X, D)
r2_insample = ...   # TODO: mD_full.score(X, D)

# print(f'in-sample R^2 of E[D|X] forest = {r2_insample:.3f}')
# print('Irreducible noise means an HONEST R^2 should be well below 1.')

### ✅ Solution 5.1

In [ ]:
r2_insample = mD_full.score(X, D)
print(f'in-sample R^2 of E[D|X] forest = {r2_insample:.3f}')
assert r2_insample > 0.8, 'in-sample R^2 is inflated by overfitting'
print('The forest explains far more in-sample variance than is real:\n'
      'that inflation is exactly the leakage cross-fitting removes.')

## 6 · DML cannot rescue unmeasured confounding

DML is an *estimation* tool, not an *identification* tool. If a confounder is not in `X`, the orthogonal score is built around the wrong nuisances and `θ̂` is biased — no matter how flexible the learner or how careful the cross-fitting. We demonstrate by **hiding column `X0`** (a strong confounder) from the learner.

In [ ]:
X_obs = X[:, 1:]   # drop X0 -- now it is an UNMEASURED confounder

theta_unm, se_unm, _, _ = dml_plm(X_obs, D, Y, crossfit=True)
print(f'DML with X0 measured   = {theta_hat:.3f}  (recovers ~0.8)')
print(f'DML with X0 UNMEASURED = {theta_unm:.3f}  (biased)')
print(f'TRUE theta             = {TRUE_THETA:.3f}')
assert abs(theta_unm - TRUE_THETA) > 0.2, 'unmeasured confounding biases DML'
print('\nThe orthogonal, cross-fitted machinery is intact -- but X0 is not\n'
      'in X, so its confounding leaks straight back into theta_hat.')

### 🔧 Exercise 6.1 — does a fancier learner help?

A natural hope: maybe a *deeper, bigger* forest can compensate for the missing confounder. Re-run the unmeasured-confounding case with a more flexible learner (more trees) and check whether the bias goes away. Predict the answer before running.

Complete the `# TODO`.

In [ ]:
# TODO: run DML on X_obs (X0 still hidden) with a bigger forest.
theta_big = ...   # TODO: dml_plm(X_obs, D, Y, crossfit=True,
                  #                n_estimators=300)[0]
# print(f'bigger-forest theta (X0 hidden) = {theta_big:.3f}')
# print('More flexibility cannot conjure a confounder that is not in X.')

### ✅ Solution 6.1

In [ ]:
theta_big = dml_plm(X_obs, D, Y, crossfit=True,
                    n_estimators=300)[0]
print(f'standard forest (X0 hidden) = {theta_unm:.3f}')
print(f'bigger forest   (X0 hidden) = {theta_big:.3f}')
print(f'TRUE theta                  = {TRUE_THETA:.3f}')
assert abs(theta_big - TRUE_THETA) > 0.2, 'still biased -- flexibility cannot help'
print('\nConfirmed: a fancier learner does NOT fix unmeasured confounding.\n'
      'Identification is an assumption, not something ML can estimate away.')

## 7 · Wrap-up & self-check

- The **partially linear model** `Y = θD + g(X) + ε`, `D = m(X) + v` isolates one effect `θ` amid flexible, high-dimensional nuisances.
- **Naive plug-in ML is biased**: regularization bias and overfitting leak into `θ̂`. *Prediction is not estimation.*
- **Frisch–Waugh–Lovell** gives the cure's skeleton: residualize `Y` and `D` on `X`, then regress the residuals. DML swaps OLS partialling for ML partialling.
- **Neyman orthogonality** makes the score first-order insensitive to nuisance error, so slow ML rates suffice for a √n-normal `θ̂`.
- **Cross-fitting** removes own-observation overfitting; skip it and the in-sample residual variances collapse and the SE is invalid.
- **DML does NOT fix** unmeasured confounding (we saw `θ̂` jump when we hid `X0`), positivity, or a wrong graph.

**You're ready for Week 13** if you can build the cross-fitted residual-on-residual estimator from memory, explain why it beats the naive plug-in, and state one thing DML cannot do. Next week: heterogeneous effects and policy learning — turning one average effect into 'for whom, and what should we do?'